<a href="https://colab.research.google.com/github/Ksa88349/Kareem-Data-Analysis-Portfolio/blob/main/Customer_Data/E_Commerce_Customer_Cleaning_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Details — Data Cleaning

This notebook documents the data-quality assessment, cleaning decisions, validation, and export of the `customer_details` table.

The original `customer_age` column is preserved, while `customer_age_analysis` is created for analysis after handling unrealistic age values.


## 1. Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np

cus_det = pd.read_csv("/content/customer_details.csv")


## 2. Initial Data Inspection

In [2]:
print("Shape:", cus_det.shape)

cus_det.info()
cus_det.head()


Shape: (20000, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   customer_id   20000 non-null  int64  
 1   sex           20000 non-null  object 
 2   customer_age  20000 non-null  float64
 3   tenure        20000 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 625.1+ KB


,customer_id,sex,customer_age,tenure
0,9798859,Male,44.0,93
1,11413563,Male,36.0,65
2,818195,Male,35.0,129
3,12049009,Male,33.0,58
4,10083045,Male,42.0,88


## 3. Check for Duplicates and Missing Values

In [3]:
print("Duplicate rows:", cus_det.duplicated().sum())

print("\nMissing values:")
print(cus_det.isnull().sum())


Duplicate rows: 0

Missing values:
customer_id     0
sex             0
customer_age    0
tenure          0
dtype: int64


In [4]:
cus_det = cus_det.drop_duplicates()


## 4. Inspect Categorical Values

In [5]:
cus_det["sex"].value_counts()


,count
sex,
Male,15322
Female,4669
kvkktalepsilindi,8
UNKNOWN,1


### Sex Value Cleaning

An unexpected value was found in `sex` and was standardized to `UNKNOWN`.


In [6]:
cus_det["sex"] = cus_det["sex"].replace({
    "kvkktalepsilindi": "UNKNOWN"
})

cus_det["sex"].value_counts()


,count
sex,
Male,15322
Female,4669
UNKNOWN,9


## 5. Customer Age — Data Quality Assessment

In [7]:
cus_det["customer_age"].describe()


,customer_age
count,20000.000000
mean,262.222550
std,604.321589
min,-34.000000
25%,29.000000
50%,38.000000
75%,123.000000
max,2022.000000


In [8]:
print("Ages between 15 and 80:", cus_det["customer_age"].between(15, 80).sum())
print("Ages between 81 and 150:", cus_det["customer_age"].between(81, 150).sum())
print("Age = 2022:", (cus_det["customer_age"] == 2022).sum())
print("Ages below 15:", (cus_det["customer_age"] < 15).sum())
print("Negative ages:", (cus_det["customer_age"] < 0).sum())


Ages between 15 and 80: 12563
Ages between 81 and 150: 4610
Age = 2022: 2102
Ages below 15: 725
Negative ages: 1


In [9]:
cus_det.sort_values("customer_age", ascending=False).head(20)


,customer_id,sex,customer_age,tenure
13754,16924789,Male,2022.0,30
13755,16896081,Male,2022.0,30
13756,16933035,Male,2022.0,30
13766,16928499,Male,2022.0,30
13767,16883083,Male,2022.0,30
13768,16916851,Male,2022.0,30
13778,16908381,Male,2022.0,30
13779,16895229,Male,2022.0,30
13780,16931929,Male,2022.0,30
13781,16932489,Male,2022.0,30


In [10]:
cus_det.sort_values("customer_age").head(10)


,customer_id,sex,customer_age,tenure
6221,40047455,Male,-34.0,6
13387,17642583,Female,3.0,28
5967,41548409,Male,3.0,5
10710,21943943,Male,4.0,21
10888,21968835,Male,5.0,21
10964,22158339,Male,5.0,21
10965,22343741,Male,5.0,21
10966,21810887,Male,5.0,21
10967,22246049,Male,5.0,21
10852,22041145,Male,5.0,21


## 6. Investigating Invalid Ages

The `customer_age` column contains several unrealistic values. In particular:

- 2,102 records have an age value of `2022`.
- 4,610 records have ages between `81` and `150`.
- Other records contain ages below `15`, including negative values.

Because removing all invalid records would cause substantial data loss, `tenure` was investigated as a potential source for recovering a reasonable age value.


In [11]:
suspect = cus_det[
    (cus_det["customer_age"] > 80) &
    (cus_det["customer_age"] <= 150)
]

suspect["tenure"].describe()


,tenure
count,4610.000000
mean,27.385466
std,21.601481
min,4.000000
25%,13.000000
50%,22.000000
75%,36.000000
max,131.000000


In [12]:
print(
    "Invalid ages (81-150) with a reasonable tenure (15-80):",
    (
        (cus_det["customer_age"] > 80) &
        (cus_det["customer_age"] <= 150) &
        (cus_det["tenure"].between(15, 80))
    ).sum()
)

print(
    "Age = 2022 with a reasonable tenure (15-80):",
    (
        (cus_det["customer_age"] == 2022) &
        (cus_det["tenure"].between(15, 80))
    ).sum()
)


Invalid ages (81-150) with a reasonable tenure (15-80): 3153
Age = 2022 with a reasonable tenure (15-80): 2102


## 7. Age Correction Strategy

A new analytical column named `customer_age_analysis` is created while preserving the original `customer_age`.

Correction rules:

1. If `customer_age` is between `15` and `80`, keep the original age.
2. If `customer_age` is outside the valid range and `tenure` is between `15` and `80`, use `tenure` as the replacement age.
3. If both values are invalid, set the analytical age to `NaN`.

This approach reduces data loss while preserving the original source values.


In [13]:
cus_det["customer_age_analysis"] = cus_det["customer_age"]


In [14]:
cus_det.loc[
    (cus_det["customer_age_analysis"] < 15) |
    (cus_det["customer_age_analysis"] > 80),
    "customer_age_analysis"
] = np.nan


In [15]:
cus_det.loc[
    (cus_det["customer_age_analysis"].isna()) &
    (cus_det["tenure"].between(15, 80)),
    "customer_age_analysis"
] = cus_det["tenure"]


## 8. Validate the Age Correction

In [16]:
cus_det["customer_age_analysis"].describe()


,customer_age_analysis
count,18458.000000
mean,33.276628
std,9.832155
min,15.000000
25%,27.000000
50%,32.000000
75%,39.000000
max,80.000000


In [17]:
print("Missing analytical ages:", cus_det["customer_age_analysis"].isna().sum())
print(
    "Analytical ages outside 15-80:",
    (
        (cus_det["customer_age_analysis"] < 15) |
        (cus_det["customer_age_analysis"] > 80)
    ).sum()
)


Missing analytical ages: 1542
Analytical ages outside 15-80: 0


In [18]:
cus_det.loc[
    cus_det["customer_age"] == 2022,
    ["customer_age", "tenure", "customer_age_analysis"]
].head(20)


,customer_age,tenure,customer_age_analysis
13754,2022.0,30,30.0
13755,2022.0,30,30.0
13756,2022.0,30,30.0
13766,2022.0,30,30.0
13767,2022.0,30,30.0
13768,2022.0,30,30.0
13778,2022.0,30,30.0
13779,2022.0,30,30.0
13780,2022.0,30,30.0
13781,2022.0,30,30.0


### Validation Result

After the correction, `customer_age_analysis` contains only values within the selected analytical range (`15–80`) or `NaN` where the age could not be reasonably recovered.


## 9. Final Table Check

In [19]:
cus_det.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id            20000 non-null  int64  
 1   sex                    20000 non-null  object 
 2   customer_age           20000 non-null  float64
 3   tenure                 20000 non-null  int64  
 4   customer_age_analysis  18458 non-null  float64
dtypes: float64(2), int64(2), object(1)
memory usage: 781.4+ KB


In [20]:
cus_det.isnull().sum()


,0
customer_id,0
sex,0
customer_age,0
tenure,0
customer_age_analysis,1542


In [21]:
cus_det.head()


,customer_id,sex,customer_age,tenure,customer_age_analysis
0,9798859,Male,44.0,93,44.0
1,11413563,Male,36.0,65,36.0
2,818195,Male,35.0,129,35.0
3,12049009,Male,33.0,58,33.0
4,10083045,Male,42.0,88,42.0


## 10. Export Clean Dataset

In [22]:
cus_det.to_csv(
    "customer_details_clean.csv",
    index=False
)


The cleaned customer table is exported as `customer_details_clean.csv`.

The original `customer_age` column is retained for traceability, while `customer_age_analysis` is the age field intended for analysis.
